## Importing

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
data = pd.read_csv('daily-min-temp.csv')
df = data.copy()
data.head()

,Date,min_temp
0,1981-01-01,20.7
1,1981-01-02,17.9
2,1981-01-03,18.8
3,1981-01-04,14.6
4,1981-01-05,15.8


## Preprocessing

In [4]:
data['Date'] = pd.to_datetime(df['Date'])
data['min_temp'] = pd.to_numeric(data['min_temp'], errors='coerce')
data.dtypes

Date        datetime64[ns]
min_temp           float64
dtype: object

In [5]:
data.isnull().sum()

Date        0
min_temp    3
dtype: int64

In [6]:
data['min_temp'] = data['min_temp'].interpolate(method='linear')

In [7]:
data.set_index('Date', inplace=True)

In [9]:
data

,min_temp
Date,
1981-01-01,20.7
1981-01-02,17.9
1981-01-03,18.8
1981-01-04,14.6
1981-01-05,15.8
...,...
1990-12-27,14.0
1990-12-28,13.6
1990-12-29,13.5


## Visualisation

In [ ]:
plt.figure(figsize=(14,5))
plt.plot(data.index, data['min_temp'], color='blue')
plt.title('Daily Minimum Temperature Over Time')
plt.xlabel('Date')
plt.ylabel('Min Temperature')
plt.show()


In [ ]:
rolling_window = 30 
data['rolling_mean'] = data['min_temp'].rolling(window=rolling_window).mean()
data['rolling_std'] = data['min_temp'].rolling(window=rolling_window).std()

plt.figure(figsize=(14,5))
plt.plot(data['min_temp'], label='Original')
plt.plot(data['rolling_mean'], color='red', label=f'{rolling_window}-day Rolling Mean')
plt.plot(data['rolling_std'], color='green', label=f'{rolling_window}-day Rolling Std')
plt.title('Trend and Variability Check')
plt.legend()
plt.show()


Rolling mean helps identify long-term trend.

Rolling std shows periods of high/low variability.

Large fluctuations in rolling std may indicate seasonal effects.

## Decomposition

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

result = seasonal_decompose(data['min_temp'], model='additive', period=365)
result.plot()
plt.show()

In [ ]:
import seaborn as sns
data['month'] = data.index.month
data['day'] = data.index.day

monthly_avg = data.groupby('month')['min_temp'].mean()

plt.figure(figsize=(10,4))
sns.barplot(x=monthly_avg.index, y=monthly_avg.values, palette='coolwarm')
plt.title('Average Temperature by Month')
plt.xlabel('Month')
plt.ylabel('Average Min Temperature')
plt.show()

## Train Test Splitting

In [ ]:
data.tail()

In [ ]:
data = data.asfreq('D')

In [ ]:
train = data['1981':'1988']
test = data['1989':'1990']

# Modeling

## ARIMA

### Check stationarity

In [ ]:
train = train.interpolate(method='linear')

In [ ]:
from statsmodels.tsa.stattools import adfuller

result = adfuller(train['min_temp'])
result

Null hypothesis (H0): The series is non-stationary.

Alternative hypothesis (H1): The series is stationary.

p-value < 0.05 → reject H0 → series is stationary.

If p-value > 0.05 → series is non-stationary → we need differencing (d) in ARIMA.

### ACF and PACF

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

plt.figure(figsize=(14,5))
plot_acf(train['min_temp'].dropna(), lags=50)
plt.show()

plt.figure(figsize=(14,5))
plot_pacf(train['min_temp'].dropna(), lags=50)
plt.show()

ACF has spikes at lags 1 and 2 and then drops inside the confidence bands:

This suggests MA(2) → include q=2.

PACF has significant spikes at lag 1 and 2, then drops inside the confidence bands:

This suggests AR(2) → include p=2.

### Fitting

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error

start with a simple order (p=2, d=1, q=2) or use auto_arima

In [ ]:
order = (2,1,2)

model_arima = ARIMA(train['min_temp'], order=order)
model_arima_fit = model_arima.fit()

print(model_arima_fit.summary())

In [ ]:
n_test = len(test)
forecast_arima = model_arima_fit.forecast(steps=n_test)

In [ ]:
mse = mean_squared_error(test['min_temp'], forecast_arima)
rmse_arima = np.sqrt(mse)
rmse_arima

In [ ]:
forecast_arima = pd.Series(forecast_arima, index=test.index)

In [ ]:
plt.figure(figsize=(14,5))
plt.plot(train['min_temp'], label='Train')
plt.plot(test['min_temp'], label='Test', color='green')
plt.plot(forecast_arima, label='ARIMA Forecast', color='red')
plt.title('ARIMA Forecast vs Actual')
plt.xlabel('Date')
plt.ylabel('Min Temperature')
plt.legend()
plt.show()

## SARIMA

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

order = (2,1,2)
seasonal_order = (1,1,1,30)

model_sarima = SARIMAX(train['min_temp'],
                      order=order,
                      seasonal_order=seasonal_order,
                      enforce_stationarity=False,
                      enforce_invertibility=False)
model_sarima_fit = model_sarima.fit()
print(model_sarima_fit.summary())

In [ ]:
n_test = len(test)
forecast_sarima_values = model_sarima_fit.forecast(steps=n_test)
forecast_sarima = pd.Series(forecast_sarima_values, index=test.index)

In [ ]:
plt.figure(figsize=(14,5))
plt.plot(train['min_temp'], label='Train')
plt.plot(test['min_temp'], label='Test', color='green')
plt.plot(forecast_sarima, label='SARIMA Forecast', color='red')
plt.title('SARIMA Forecast vs Actual')
plt.xlabel('Date')
plt.ylabel('Min Temperature')
plt.legend()
plt.show()

In [ ]:
mse = mean_squared_error(test['min_temp'], forecast_sarima)
rmse_sarima = np.sqrt(mse)
rmse_sarima

SARIMA with daily data and yearly seasonality (m=365) is computationally expensive. Using weekly seasonality (m=7) is faster but doesn’t fully capture yearly patterns. For such daily datasets, Holt-Winters or Prophet are more practical alternatives for forecasting trend and seasonality efficiently

## Exponential Smoothing (Holt-Winters)

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

In [ ]:
model_hw = ExponentialSmoothing(
    train['min_temp'],
    trend='add',
    seasonal='add',
    seasonal_periods=365
)
model_hw_fit = model_hw.fit()

In [ ]:
n_test = len(test)
forecast_hw = model_hw_fit.forecast(steps=n_test)
forecast_hw = pd.Series(forecast_hw, index=test.index)

In [ ]:
plt.figure(figsize=(14,5))
plt.plot(train['min_temp'], label='Train')
plt.plot(test['min_temp'], label='Test', color='green')
plt.plot(forecast_hw, label='Holt-Winters Forecast', color='red')
plt.title('Holt-Winters Forecast vs Actual')
plt.xlabel('Date')
plt.ylabel('Min Temperature')
plt.legend()
plt.show()

In [ ]:
rmse_hw = np.sqrt(mean_squared_error(test['min_temp'], forecast_hw))
print("Holt-Winters RMSE:", rmse_hw)

## Prophet

In [ ]:
from prophet import Prophet

Prophet expects a DataFrame with columns ds (datetime) and y (target):

In [ ]:
prophet_train = train.reset_index()[['Date', 'min_temp']].rename(columns={'Date': 'ds', 'min_temp': 'y'})
prophet_test = test.reset_index()[['Date', 'min_temp']].rename(columns={'Date': 'ds', 'min_temp': 'y'})

In [ ]:
model_prophet = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False
)

model_prophet.fit(prophet_train)

In [ ]:
future = prophet_test[['ds']].copy()

forecast_prophet = model_prophet.predict(future)

In [ ]:
forecast_values = forecast_prophet['yhat'].values
forecast_prophet_series = pd.Series(forecast_values, index=test.index)

In [ ]:
plt.figure(figsize=(14,5))
plt.plot(train['min_temp'], label='Train')
plt.plot(test['min_temp'], label='Test', color='green')
plt.plot(forecast_prophet_series, label='Prophet Forecast', color='red')
plt.title('Prophet Forecast vs Actual')
plt.xlabel('Date')
plt.ylabel('Min Temperature')
plt.legend()
plt.show()

In [ ]:
rmse_prophet = np.sqrt(mean_squared_error(test['min_temp'], forecast_prophet_series))
print("Prophet RMSE:", rmse_prophet)

## LSTM

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from sklearn.metrics import mean_squared_error

In [ ]:
scaler = MinMaxScaler(feature_range=(0,1))
train_scaled = scaler.fit_transform(train[['min_temp']])
test_scaled = scaler.fit_transform(test[['min_temp']])

In [ ]:
def create_sequences(data, seq_length):
    X,y = [],[]
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length])
        y.append(data[i+seq_length])
    return np.array(X), np.array(y)

seq_length = 30
X_train, y_train = create_sequences(train_scaled, seq_length)
X_test, y_test = create_sequences(np.concatenate((train_scaled[-seq_length:], test_scaled)), seq_length)

In [ ]:
model = Sequential()
model.add(LSTM(50, activation='tanh', input_shape=(seq_length,1)))
model.add(Dense(1))
model.compile(optimizer='adam', loss='mse')

model.summary()

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=16,
    validation_split=0.2,
    verbose=1
)

In [ ]:
y_pred_scaled = model.predict(X_test)
y_pred = scaler.inverse_transform(y_pred_scaled)
y_test_actual = scaler.inverse_transform(y_test)

In [ ]:
plt.figure(figsize=(14,5))
plt.plot(test.index[:len(y_test_actual)], y_test_actual, label='Actual', color='green')
plt.plot(test.index[:len(y_pred)], y_pred, label='LSTM Forecast', color='red')
plt.title('LSTM Forecast vs Actual')
plt.xlabel('Date')
plt.ylabel('Min Temperature')
plt.legend()
plt.show()

In [ ]:
rmse_lstm = np.sqrt(mean_squared_error(y_test_actual, y_pred))
print("LSTM RMSE:", rmse_lstm)

## Comparison

In [ ]:
rmse_dict = {
    "Model": ["ARIMA", "SARIMA", "Holt-Winters", "Prophet", "LSTM"],
    "RMSE": [rmse_arima, rmse_sarima, rmse_hw, rmse_prophet, rmse_lstm]
}

rmse_table = pd.DataFrame(rmse_dict)
rmse_table

### Conclusion: 
Among all models, LSTM achieved the lowest RMSE (2.22) indicating they capture the patterns in daily minimum temperature best. 
- Traditional time series models like Holt-Winters, SARIMA, and ARIMA performed reasonably but were less accurate
- likely because they struggle with long-term trends and complex seasonality in daily data.